# Lab 3: RAG Chat Flow Implementation

**Duration:** 35 minutes  
**Environment:** Azure ML Studio Notebook  

---

## Learning Objectives

- Build a RAG pipeline using Python (simulating Prompt Flow nodes)
- Implement query rewriting for multi-turn conversations
- Connect to Azure AI Search index from Lab 2
- Generate grounded responses with source citations

## Prerequisites

Lab 1 + Lab 2 completed

---

© 2026 Great Learning. All rights reserved.

## Step 1: Load Configuration

In [ ]:
import os, json, urllib.request, ssl, subprocess

RESOURCE_GROUP = 'rg-promptflow-rag-lab'

# Auto-detect OpenAI resource that has BOTH gpt-4o and text-embedding-3-small deployments
# This handles the case where Lab 1 was run multiple times creating multiple OpenAI resources
print('Detecting Azure resources...')
r = subprocess.run(['az', 'cognitiveservices', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', "[?kind=='OpenAI'].name", '-o', 'tsv'], capture_output=True, text=True)
openai_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]

OPENAI_NAME = ''
for candidate in openai_candidates:
    # Check if this resource has both required deployments
    r = subprocess.run(['az', 'cognitiveservices', 'account', 'deployment', 'list',
        '-n', candidate, '-g', RESOURCE_GROUP, '--query', '[].name', '-o', 'tsv'],
        capture_output=True, text=True)
    deployments = r.stdout.strip().split('\n')
    if 'gpt-4o' in deployments and 'text-embedding-3-small' in deployments:
        OPENAI_NAME = candidate
        print(f'  Found OpenAI with deployments: {candidate}')
        break

if not OPENAI_NAME:
    print('ERROR: No OpenAI resource found with both gpt-4o and text-embedding-3-small deployments.')
    print(f'  Candidates checked: {openai_candidates}')
    print('  Fix: Go back to Lab 1 and run Steps 4 + 5 to create deployments.')
    raise SystemExit(1)

# Auto-detect Search service
r = subprocess.run(['az', 'search', 'service', 'list', '-g', RESOURCE_GROUP,
    '--query', '[].name', '-o', 'tsv'], capture_output=True, text=True)
search_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]
SEARCH_NAME = search_candidates[0] if search_candidates else ''

# Auto-detect Storage account
r = subprocess.run(['az', 'storage', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', '[0].name', '-o', 'tsv'], capture_output=True, text=True)
STORAGE_NAME = r.stdout.strip()

if not SEARCH_NAME:
    print('ERROR: No Search service found. Run Lab 1 first.')
    raise SystemExit(1)

# Get endpoints and keys
r = subprocess.run(['az', 'cognitiveservices', 'account', 'show', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'properties.endpoint', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_ENDPOINT = r.stdout.strip()
if not OPENAI_ENDPOINT.endswith('/'): OPENAI_ENDPOINT += '/'

r = subprocess.run(['az', 'cognitiveservices', 'account', 'keys', 'list', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'key1', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_KEY = r.stdout.strip()

SEARCH_ENDPOINT = f'https://{SEARCH_NAME}.search.windows.net'
r = subprocess.run(['az', 'search', 'admin-key', 'show', '--service-name', SEARCH_NAME, '-g', RESOURCE_GROUP,
    '--query', 'primaryKey', '-o', 'tsv'], capture_output=True, text=True)
SEARCH_KEY = r.stdout.strip()

ctx = ssl.create_default_context()

print(f'\n✅ Resources detected:')
print(f'  OpenAI:  {OPENAI_NAME}')
print(f'  Search:  {SEARCH_NAME}')
print(f'  Storage: {STORAGE_NAME}')
print(f'  Endpoint: {OPENAI_ENDPOINT}')

## Step 2: Build RAG Chat Flow — Define Pipeline Nodes

The RAG Chat Flow has 5 nodes:
1. **Query Rewriting** — rewrites follow-up questions into standalone queries
2. **Embedding** — generates vector for the query
3. **Search** — hybrid search against Azure AI Search
4. **Format** — structures retrieved documents as context
5. **Generate** — produces grounded response with citations

In [ ]:
# ── NODE 1: Query Rewriting ──
def rewrite_query(question, chat_history):
    """Rewrite follow-up question using chat history for context."""
    if not chat_history:
        return question
    history_text = '\n'.join([f"User: {h['user']}\nAssistant: {h['assistant']}" for h in chat_history[-3:]])
    prompt = f"""Given the chat history and latest question, rewrite as a standalone question.

Chat History:
{history_text}

Latest Question: {question}

Rewritten standalone question:"""
    url = f"{OPENAI_ENDPOINT}openai/deployments/gpt-4o/chat/completions?api-version=2024-06-01"
    data = json.dumps({'messages':[
        {'role':'system','content':'You rewrite questions to be standalone. Output only the rewritten question.'},
        {'role':'user','content':prompt}],'max_tokens':100,'temperature':0}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())['choices'][0]['message']['content'].strip()

# ── NODE 2: Generate Embedding ──
def embed_query(text):
    url = f"{OPENAI_ENDPOINT}openai/deployments/text-embedding-3-small/embeddings?api-version=2024-06-01"
    data = json.dumps({'input': text}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())['data'][0]['embedding']

# ── NODE 3: Search Azure AI Search ──
def search_index(query_text, query_vector, top_k=3):
    url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/docs/search?api-version=2024-07-01"
    data = json.dumps({'search':query_text,
        'vectorQueries':[{'vector':query_vector,'k':top_k,'fields':'content_vector','kind':'vector'}],
        'select':'title,content,category,source_file','top':top_k}).encode()
    req = urllib.request.Request(url, data=data, method='POST',
        headers={'Content-Type':'application/json','api-key':SEARCH_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read()).get('value', [])

# ── NODE 4: Format Documents ──
def format_documents(search_results):
    context_parts, sources = [], []
    for i, doc in enumerate(search_results, 1):
        context_parts.append(f"[Source {i}: {doc['title']}]\n{doc['content']}")
        sources.append({'id':i,'title':doc['title'],'file':doc['source_file']})
    return '\n\n---\n\n'.join(context_parts), sources

# ── NODE 5: Generate Response ──
def generate_response(question, context, chat_history):
    system_prompt = """You are a SecureBank banking policy assistant.
Rules:
- ONLY answer based on the provided context documents
- Always cite sources using [Source N] format
- If context doesn't contain the answer, say so
- Never provide personal financial advice
- Be concise and accurate"""
    messages = [{'role':'system','content':system_prompt}]
    for h in chat_history[-3:]:
        messages.append({'role':'user','content':h['user']})
        messages.append({'role':'assistant','content':h['assistant']})
    messages.append({'role':'user','content':f'Context:\n{context}\n\nQuestion: {question}'})
    url = f"{OPENAI_ENDPOINT}openai/deployments/gpt-4o/chat/completions?api-version=2024-06-01"
    data = json.dumps({'messages':messages,'max_tokens':150,'temperature':0.1}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())['choices'][0]['message']['content']

print('✅ All 5 RAG pipeline nodes defined.')

## Step 3: Complete RAG Chat Flow Pipeline

This function chains all 5 nodes together into a single pipeline call.

In [ ]:
def rag_chat(question, chat_history=None):
    """Execute the complete RAG Chat Flow pipeline."""
    if chat_history is None: chat_history = []
    print(f'\n{"="*60}')
    print(f'👤 User: {question}')
    print(f'{"="*60}')
    
    # Node 1: Rewrite query
    standalone = rewrite_query(question, chat_history)
    if standalone != question: print(f'🔄 Rewritten: {standalone}')
    
    # Node 2: Embed
    vector = embed_query(standalone)
    print(f'🔢 Embedding generated ({len(vector)} dimensions)')
    
    # Node 3: Search
    results = search_index(standalone, vector)
    print(f'🔍 Found {len(results)} relevant documents')
    
    # Node 4: Format
    context, sources = format_documents(results)
    for s in sources: print(f'   📄 [{s["id"]}] {s["title"]} ({s["file"]})')
    
    # Node 5: Generate
    response = generate_response(question, context, chat_history)
    print(f'\n🤖 Assistant: {response}')
    print(f'{"="*60}')
    return response

print('✅ RAG Chat Flow pipeline ready.')

## Step 4: Test — Multi-Turn Banking Conversation

Watch how the pipeline handles context across turns. Turn 2 ("What about business accounts?") requires understanding from Turn 1.

In [ ]:
print('🏦 ' * 20)
print('  BANKING RAG CHAT FLOW — DEMO')
print('🏦 ' * 20)

chat_history = []

# Turn 1
q1 = 'What is the wire transfer limit for personal accounts?'
a1 = rag_chat(q1, chat_history)
chat_history.append({'user': q1, 'assistant': a1})

# Turn 2 — requires context from Turn 1
q2 = 'What about business accounts?'
a2 = rag_chat(q2, chat_history)
chat_history.append({'user': q2, 'assistant': a2})

# Turn 3
q3 = 'How do I report a fraudulent transaction?'
a3 = rag_chat(q3, chat_history)
chat_history.append({'user': q3, 'assistant': a3})

# Turn 4
q4 = 'What savings account has the highest interest rate?'
a4 = rag_chat(q4, chat_history)
chat_history.append({'user': q4, 'assistant': a4})

print('\n✅ RAG Chat Flow demo complete — 4 turns with context awareness.')

## 📋 Bonus: Using Azure AI Foundry Studio UI

To use the visual Prompt Flow editor:

1. Go to https://ai.azure.com
2. Select your project (or create one in your resource group)
3. Navigate to **Prompt Flow** in the left menu
4. Click **+ Create** → **Chat Flow**
5. Select the *Chat with Wikipedia* or *Multi-Round QA* template
6. Modify the flow:
   - Replace Wikipedia lookup with Azure AI Search connection
   - Update the system prompt for banking domain
   - Connect your Azure OpenAI deployment (gpt-4o)
   - Set the search index to `banking-policies`
7. Click **Run** to test the flow interactively

## ✅ Lab 3 Complete

**What you accomplished:**
- Built a complete RAG Chat Flow pipeline (5 nodes)
- Implemented query rewriting for multi-turn conversations
- Connected to Azure AI Search with hybrid search
- Generated grounded responses with source citations
- Tested 4-turn banking conversation

**Next:** Open `promptflow_lab4_conversational_search.ipynb`


> 🧹 **Cleanup:** Run the cleanup cell in **Lab 9** when all labs are complete to delete all resources.
---

© 2026 Great Learning. All rights reserved.